# 📦 S3 Object Staleness Predictor

**Objective:** Build a binary classification model to predict whether an S3 inventory object is stale (`is_stale = True`) using object metadata including access history.

**Dataset:** `data/inventory_updated.csv` — 92,369 records, 10 columns.

**Target:** `is_stale` (Boolean — 1 = Stale, 0 = Active)

**New in this version:** The dataset now includes a `last_accessed` date column, which is expected to be one of the strongest staleness predictors.

---
## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Load & Inspect Data](#2-load--inspect-data)
3. [Data Cleaning](#3-data-cleaning)
4. [Exploratory Data Analysis (EDA)](#4-exploratory-data-analysis-eda)
5. [Outlier Analysis](#5-outlier-analysis)
6. [Feature Engineering](#6-feature-engineering)
7. [Preprocessing & Class Imbalance](#7-preprocessing--class-imbalance)
8. [Baseline Model — Logistic Regression](#8-baseline-model--logistic-regression)
9. [Improved Model — Random Forest](#9-improved-model--random-forest)
10. [Model Evaluation & Comparison](#10-model-evaluation--comparison)
11. [Feature Importance](#11-feature-importance)
12. [Predict on New Objects](#12-predict-on-new-objects)
13. [Conclusion](#13-conclusion)

---
## 1. Setup & Imports

In [ ]:
# ── Standard library ───────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Scikit-learn: preprocessing & model selection ──────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ── Scikit-learn: models ───────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ── Scikit-learn: evaluation ───────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    f1_score
)

# ── Class imbalance ────────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

# ── Global plot settings ───────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 5)})

RANDOM_STATE = 42
DATA_PATH    = 'data/inventory_updated.csv'

print('✅ All libraries imported successfully.')

---
## 2. Load & Inspect Data

We load the CSV, parse both date columns (`last Modified` and `last accessed column`) at read time, and standardise all column names. A first-pass inspection covers shape, dtypes, missing values, and class distribution.

In [ ]:
# Load CSV — parse both date columns immediately for efficiency
df = pd.read_csv(
    DATA_PATH,
    parse_dates=['last Modified', 'last accessed column'],
    low_memory=False
)

# Normalise column names: strip whitespace, lowercase, underscores
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)

# Rename verbose column to concise names
df = df.rename(columns={
    'last_modified':        'last_modified',
    'last_accessed_column': 'last_accessed'
})

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('\nColumn dtypes:')
print(df.dtypes)
print('\nFirst 3 rows:')
df.head(3)

In [ ]:
# Summarise missing values and cardinality per column
summary = pd.DataFrame({
    'dtype':     df.dtypes,
    'missing':   df.isna().sum(),
    'missing_%': (df.isna().mean() * 100).round(2),
    'unique':    df.nunique()
})
print(summary.to_string())

In [ ]:
# Class balance of the target variable
class_counts = df['is_stale'].value_counts()
class_pct    = df['is_stale'].value_counts(normalize=True) * 100
print('Target class distribution:')
print(pd.DataFrame({'count': class_counts, '%': class_pct.round(2)}))

---
## 3. Data Cleaning

Steps performed in order:
1. **Remove exact duplicate rows** — identical records add no information.
2. **Drop rows with missing target** — cannot train without a label.
3. **Impute categorical nulls** — `version_id` and `encryption_type` have domain-meaningful fill values.
4. **Impute numeric nulls** — `size` uses median imputation; negative/zero values are invalid.
5. **Impute missing dates** — `last_accessed` may be null for objects never accessed; flagged separately.
6. **Cast booleans to int** — required for scikit-learn compatibility.

In [ ]:
# ── Step 1: Remove duplicate rows ─────────────────────────────────────────────
n_before = len(df)
df = df.drop_duplicates()
print(f'Duplicates removed : {n_before - len(df):,}')
print(f'Rows remaining     : {len(df):,}')

In [ ]:
# ── Step 2: Drop rows with no label ───────────────────────────────────────────
df = df.dropna(subset=['is_stale'])

# ── Step 3: Impute categorical nulls ──────────────────────────────────────────
# Missing version_id → object has no versioning
df['version_id'] = df['version_id'].fillna('none')
# Missing encryption_type → object is unencrypted
df['encryption_type'] = df['encryption_type'].fillna('NONE')

# ── Step 4: Impute numeric nulls ───────────────────────────────────────────────
# Replace invalid sizes (<=0) with NaN, then median-impute
df.loc[df['size'] <= 0, 'size'] = np.nan
median_size = df['size'].median()
df['size'] = df['size'].fillna(median_size)

# ── Step 5: Handle missing last_modified (drop — required for time features) ──
df = df.dropna(subset=['last_modified'])

# ── Step 6: Flag and fill missing last_accessed ────────────────────────────────
# never_accessed=1 is itself a strong staleness signal
df['never_accessed'] = df['last_accessed'].isna().astype(int)
# Fill missing last_accessed with last_modified (conservative lower bound)
df['last_accessed'] = df['last_accessed'].fillna(df['last_modified'])

# ── Step 7: Cast booleans to int ───────────────────────────────────────────────
bool_cols = ['is_latest', 'delete_marker', 'is_stale']
df[bool_cols] = df[bool_cols].astype(int)

print(f'Clean shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Total nulls : {df.isna().sum().sum()}')

---
## 4. Exploratory Data Analysis (EDA)

We explore:
- Target class balance
- Continuous variable distributions (`size`)
- Boolean and categorical features vs. staleness
- Access and modification date patterns
- Bucket-level staleness rates (interactive)

In [ ]:
# ── Target class distribution ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = ['Active (0)', 'Stale (1)']
counts = df['is_stale'].value_counts().sort_index().values
colors = ['#4C72B0', '#DD8452']

# Bar chart with count annotations
axes[0].bar(labels, counts, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution: is_stale', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(counts):
    axes[0].text(i, v + 150, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts, labels=labels, autopct='%1.1f%%',
            colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Class Proportion', fontweight='bold')

plt.suptitle('Target Variable Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Object size distribution by staleness ──────────────────────────────────────
df['size_mb'] = df['size'] / (1024 ** 2)   # convert bytes → MB

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlaid histograms on log scale (size is heavily right-skewed)
for val, lbl, col in [(0, 'Active', '#4C72B0'), (1, 'Stale', '#DD8452')]:
    axes[0].hist(
        np.log1p(df.loc[df['is_stale'] == val, 'size_mb']),
        bins=50, alpha=0.6, label=lbl, color=col, edgecolor='none'
    )
axes[0].set_title('Object Size Distribution by Staleness\n(log scale)', fontweight='bold')
axes[0].set_xlabel('log(Size + 1) in MB')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Box plot: median & spread per class (outliers hidden for clarity)
sns.boxplot(data=df, x='is_stale', y='size_mb',
            palette={'0': '#4C72B0', '1': '#DD8452'},
            showfliers=False, ax=axes[1])
axes[1].set_title('Size (MB) by Class\n(outliers hidden)', fontweight='bold')
axes[1].set_xlabel('is_stale  (0 = Active, 1 = Stale)')
axes[1].set_ylabel('Size (MB)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Boolean features vs staleness: subplots ────────────────────────────────────
bool_features = ['is_latest', 'delete_marker', 'never_accessed']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, bool_features):
    ct = df.groupby([col, 'is_stale']).size().unstack(fill_value=0)
    ct.index = ct.index.map({0: 'False', 1: 'True'})
    ct.columns = ['Active', 'Stale']
    ct.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'],
            edgecolor='white', width=0.6, rot=0)
    ax.set_title(f'{col}\nvs Staleness', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.legend(title='is_stale')

plt.suptitle('Boolean Features vs Staleness', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Encryption type vs staleness — interactive Plotly bar ──────────────────────
enc_df = (
    df.groupby(['encryption_type', 'is_stale'])
    .size()
    .reset_index(name='count')
)
enc_df['is_stale'] = enc_df['is_stale'].map({0: 'Active', 1: 'Stale'})

fig = px.bar(
    enc_df,
    x='encryption_type', y='count', color='is_stale',
    barmode='group',
    color_discrete_map={'Active': '#4C72B0', 'Stale': '#DD8452'},
    title='Encryption Type vs Staleness',
    labels={'encryption_type': 'Encryption Type',
            'count': 'Object Count', 'is_stale': 'Status'}
)
fig.update_layout(xaxis_tickangle=-30, plot_bgcolor='white')
fig.show()

In [ ]:
# ── Last accessed vs last modified: temporal overview ──────────────────────────
# Compute a temporary column for plotting only
tmp = df[['last_modified', 'last_accessed', 'is_stale']].copy()

# Ensure UTC-aware timestamps
for col in ['last_modified', 'last_accessed']:
    if tmp[col].dt.tz is None:
        tmp[col] = tmp[col].dt.tz_localize('UTC')

ref_date = pd.Timestamp.now(tz='UTC').normalize()
tmp['days_since_modified'] = (ref_date - tmp['last_modified']).dt.days
tmp['days_since_accessed'] = (ref_date - tmp['last_accessed']).dt.days

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE: days since last access by class
for val, lbl, col in [(0, 'Active', '#4C72B0'), (1, 'Stale', '#DD8452')]:
    sns.kdeplot(
        tmp.loc[tmp['is_stale'] == val, 'days_since_accessed'],
        ax=axes[0], label=lbl, color=col, fill=True, alpha=0.4
    )
axes[0].set_title('Days Since Last Accessed\nby Staleness', fontweight='bold')
axes[0].set_xlabel('Days Since Last Access')
axes[0].set_ylabel('Density')
axes[0].legend(title='is_stale')

# KDE: days since last modified by class
for val, lbl, col in [(0, 'Active', '#4C72B0'), (1, 'Stale', '#DD8452')]:
    sns.kdeplot(
        tmp.loc[tmp['is_stale'] == val, 'days_since_modified'],
        ax=axes[1], label=lbl, color=col, fill=True, alpha=0.4
    )
axes[1].set_title('Days Since Last Modified\nby Staleness', fontweight='bold')
axes[1].set_xlabel('Days Since Last Modified')
axes[1].set_ylabel('Density')
axes[1].legend(title='is_stale')

plt.suptitle('Temporal Features vs Staleness', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 15 buckets by stale count — interactive Plotly horizontal bar ──────────
top_buckets = (
    df[df['is_stale'] == 1]
    .groupby('bucket')
    .size()
    .nlargest(15)
    .reset_index(name='stale_count')
)

fig = px.bar(
    top_buckets,
    x='stale_count', y='bucket', orientation='h',
    color='stale_count', color_continuous_scale='OrRd',
    title='Top 15 Buckets by Stale Object Count',
    labels={'stale_count': 'Stale Object Count', 'bucket': 'Bucket'}
)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='white', coloraxis_showscale=False
)
fig.show()

---
## 5. Outlier Analysis

We apply the **IQR method** to detect anomalies in `size_mb` — the only continuous numeric feature. Extreme values are **Winsorized** (clipped to the 1st/99th percentiles) to prevent distortion without removing records.

In [ ]:
# ── IQR outlier detection on size_mb ──────────────────────────────────────────
Q1, Q3  = df['size_mb'].quantile([0.25, 0.75])
IQR     = Q3 - Q1
lo, hi  = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outlier_mask = (df['size_mb'] < lo) | (df['size_mb'] > hi)
print(f'Outliers (IQR): {outlier_mask.sum():,}  ({outlier_mask.mean()*100:.1f}%)')
print(f'Bounds: [{lo:.4f}, {hi:.4f}] MB   |   Max observed: {df["size_mb"].max():.2f} MB')

In [ ]:
# ── Visualise outliers ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
axes[0].boxplot(df['size_mb'], patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.6))
axes[0].set_title('Size (MB) — Box Plot', fontweight='bold')
axes[0].set_ylabel('Size (MB)')
axes[0].set_xticks([])

# Scatter: normal vs outlier
rng = np.random.default_rng(RANDOM_STATE)
normal_idx  = df.index[~outlier_mask]
outlier_idx = df.index[outlier_mask]
axes[1].scatter(
    rng.integers(0, len(df), len(normal_idx)),
    df.loc[normal_idx, 'size_mb'],
    alpha=0.15, s=2, color='#4C72B0', label='Normal'
)
axes[1].scatter(
    rng.integers(0, len(df), len(outlier_idx)),
    df.loc[outlier_idx, 'size_mb'],
    alpha=0.5, s=8, color='#DD8452',
    label=f'Outlier (n={len(outlier_idx):,})'
)
axes[1].set_title('Size (MB) — Outlier Scatter', fontweight='bold')
axes[1].set_xlabel('Record Index (approximate)')
axes[1].set_ylabel('Size (MB)')
axes[1].legend()

plt.suptitle('Outlier Analysis: Object Size', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Winsorise size_mb at the 1st / 99th percentiles ───────────────────────────
p01 = df['size_mb'].quantile(0.01)
p99 = df['size_mb'].quantile(0.99)
df['size_mb'] = df['size_mb'].clip(lower=p01, upper=p99)
print(f'size_mb clipped to [{p01:.4f}, {p99:.2f}] MB')

---
## 6. Feature Engineering

Raw metadata needs transformation into model-ready features. The `last_accessed` column unlocks several powerful predictors:

| New Feature | Source | Description |
|---|---|---|
| `days_since_modified` | `last_modified` | Days from last write to today |
| `days_since_accessed` | `last_accessed` | Days from last read to today — **key recency signal** |
| `access_to_modify_gap` | both dates | Days between last access and last modification — objects accessed long after modification may be archived |
| `modified_year` / `modified_month` | `last_modified` | Calendar decomposition for seasonal patterns |
| `accessed_year` / `accessed_month` | `last_accessed` | Calendar decomposition |
| `never_accessed` | imputation flag | 1 if `last_accessed` was originally null |
| `has_encryption` | `encryption_type` | Binary: encrypted vs not |
| `has_version` | `version_id` | Binary: versioned object |
| `size_log` | `size_mb` | Log-transformed size to reduce right skew |
| `enc_type_encoded` | `encryption_type` | Label-encoded encryption type |

In [ ]:
# ── Timezone-aware reference date ──────────────────────────────────────────────
ref_date = pd.Timestamp.now(tz='UTC').normalize()

# Ensure both date columns are UTC-aware
for col in ['last_modified', 'last_accessed']:
    if df[col].dt.tz is None:
        df[col] = df[col].dt.tz_localize('UTC')

# ── Time-delta features ────────────────────────────────────────────────────────
df['days_since_modified'] = (ref_date - df['last_modified']).dt.days
df['days_since_accessed'] = (ref_date - df['last_accessed']).dt.days

# Gap between last access and last modification (can be negative if accessed before modified)
df['access_to_modify_gap'] = (df['last_accessed'] - df['last_modified']).dt.days

# ── Calendar decomposition ─────────────────────────────────────────────────────
df['modified_year']   = df['last_modified'].dt.year
df['modified_month']  = df['last_modified'].dt.month
df['accessed_year']   = df['last_accessed'].dt.year
df['accessed_month']  = df['last_accessed'].dt.month

# ── Binary flags ───────────────────────────────────────────────────────────────
df['has_encryption'] = (df['encryption_type'] != 'NONE').astype(int)
df['has_version']    = (df['version_id'] != 'none').astype(int)

# ── Log-transform size ─────────────────────────────────────────────────────────
df['size_log'] = np.log1p(df['size_mb'])

# ── Encode encryption type ─────────────────────────────────────────────────────
le = LabelEncoder()
df['enc_type_encoded'] = le.fit_transform(df['encryption_type'])

eng_cols = [
    'days_since_modified', 'days_since_accessed', 'access_to_modify_gap',
    'modified_year', 'modified_month', 'accessed_year', 'accessed_month',
    'has_encryption', 'has_version', 'never_accessed', 'size_log', 'enc_type_encoded'
]
print('Engineered feature summary:')
print(df[eng_cols].describe().round(2).T[['mean', 'min', 'max']])

In [ ]:
# ── Stale rate by modification year and access year ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, yr_col, title in zip(
    axes,
    ['modified_year', 'accessed_year'],
    ['Stale Rate by Modification Year', 'Stale Rate by Last-Access Year']
):
    yr_rate = df.groupby(yr_col)['is_stale'].mean().reset_index()
    ax.bar(yr_rate[yr_col], yr_rate['is_stale'],
           color='#DD8452', edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Stale Rate')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.suptitle('Temporal Staleness Patterns', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap of all model features vs target ────────────────────────
heatmap_cols = [
    'size_log', 'days_since_modified', 'days_since_accessed',
    'access_to_modify_gap', 'is_latest', 'delete_marker',
    'has_encryption', 'has_version', 'never_accessed',
    'enc_type_encoded', 'modified_month', 'accessed_month', 'is_stale'
]

corr = df[heatmap_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Preprocessing & Class Imbalance

### Evaluation Metric: F1-Score (Macro)

> **Why Macro F1?**
> If the dataset is imbalanced (more active than stale objects), **accuracy** is misleading — a model predicting *all active* could score 90%+ without identifying a single stale object.
> **Macro F1** computes the F1-score independently for each class and averages them with equal weight, penalising models that perform poorly on either class. This ensures both *precision* (not incorrectly flagging healthy objects) and *recall* (catching all genuine stale objects) are optimised.

### Handling Class Imbalance: SMOTE
**SMOTE** (Synthetic Minority Oversampling Technique) is applied **on the training set only** to prevent data leakage into the test set. SMOTE creates synthetic minority-class samples by interpolating between existing ones, rather than simply duplicating rows.

In [ ]:
# ── Define feature set and target ──────────────────────────────────────────────
FEATURES = [
    'size_log',
    'days_since_modified',
    'days_since_accessed',
    'access_to_modify_gap',
    'is_latest',
    'delete_marker',
    'has_encryption',
    'has_version',
    'never_accessed',
    'enc_type_encoded',
    'modified_month',
    'accessed_month',
    'modified_year'
]

X = df[FEATURES].copy()
y = df['is_stale'].copy()

# ── Stratified train / test split ──────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train : {len(X_train):,} rows  |  Stale rate: {y_train.mean():.3f}')
print(f'Test  : {len(X_test):,} rows  |  Stale rate: {y_test.mean():.3f}')

In [ ]:
# ── Scale → SMOTE (training data only to prevent leakage) ─────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train only
X_test_scaled  = scaler.transform(X_test)        # apply same scale to test

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f'Post-SMOTE train size : {len(X_train_res):,}')
print(f'Post-SMOTE stale rate : {y_train_res.mean():.3f}')

---
## 8. Baseline Model — Logistic Regression

**Rationale for baseline choice:**
Logistic Regression is selected as the baseline because it is:
- **Interpretable** — coefficients directly reflect feature influence.
- **Fast to train** — suitable for 90k+ records.
- **A linear benchmark** — any non-linear model that fails to beat it is likely overfitting.
- **Probability-calibrated** — outputs meaningful confidence scores.

In [ ]:
# ── Train Logistic Regression ──────────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_res, y_train_res)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

f1_lr  = f1_score(y_test, y_pred_lr, average='macro')
auc_lr = roc_auc_score(y_test, y_prob_lr)

print('=== Logistic Regression — Baseline ===')
print(f'Macro F1-Score : {f1_lr:.4f}')
print(f'ROC-AUC        : {auc_lr:.4f}\n')
print(classification_report(y_test, y_pred_lr, target_names=['Active', 'Stale']))

---
## 9. Improved Model — Random Forest

**Rationale for Random Forest:**
- Captures **non-linear relationships** and feature interactions (e.g., `days_since_accessed` combined with `never_accessed`).
- Robust to unscaled features and mild noise.
- Provides **built-in feature importances**.
- Generalises well on tabular data without extensive hyperparameter search.

In [ ]:
# RF is scale-invariant — resample raw (unscaled) training data
X_train_res_raw, y_train_res_raw = SMOTE(random_state=RANDOM_STATE).fit_resample(
    X_train, y_train
)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    class_weight='balanced',   # additional guard against residual imbalance
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train_res_raw, y_train_res_raw)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

f1_rf  = f1_score(y_test, y_pred_rf, average='macro')
auc_rf = roc_auc_score(y_test, y_prob_rf)

print('=== Random Forest ===')
print(f'Macro F1-Score : {f1_rf:.4f}')
print(f'ROC-AUC        : {auc_rf:.4f}\n')
print(classification_report(y_test, y_pred_rf, target_names=['Active', 'Stale']))

In [ ]:
# ── 5-fold stratified cross-validation ────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(rf, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
print(f'5-Fold CV Macro F1 : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold           : {np.round(cv_scores, 4)}')

---
## 10. Model Evaluation & Comparison

In [ ]:
# ── Confusion matrices side-by-side ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Logistic Regression (Baseline)', 'Random Forest']
):
    disp = ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=['Active', 'Stale']
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontweight='bold')

plt.suptitle('Confusion Matrices — Test Set', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC curves overlaid ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for y_prob, label, color in [
    (y_prob_lr, f'Logistic Regression (AUC = {auc_lr:.3f})', '#4C72B0'),
    (y_prob_rf, f'Random Forest       (AUC = {auc_rf:.3f})', '#DD8452')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, label=label, lw=2, color=color)

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Chance')
ax.set_title('ROC Curve — Model Comparison', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary results table ──────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model':    ['Logistic Regression (Baseline)', 'Random Forest'],
    'Macro F1': [round(f1_lr, 4), round(f1_rf, 4)],
    'ROC-AUC':  [round(auc_lr, 4), round(auc_rf, 4)]
})
print(results.to_string(index=False))

---
## 11. Feature Importance

In [ ]:
# ── Static Seaborn horizontal bar chart ───────────────────────────────────────
imp_df = (
    pd.DataFrame({'feature': FEATURES, 'importance': rf.feature_importances_})
    .sort_values('importance', ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 7))
colors  = plt.cm.OrRd(np.linspace(0.3, 0.9, len(imp_df)))
bars    = ax.barh(imp_df['feature'], imp_df['importance'], color=colors)

ax.set_title('Feature Importances — Random Forest\n(Mean Gini Impurity Decrease)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.set_ylabel('Feature')

# Annotate bars with score values
for bar in bars:
    ax.text(bar.get_width() + 0.001,
            bar.get_y() + bar.get_height() / 2,
            f'{bar.get_width():.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Interactive Plotly feature importance ──────────────────────────────────────
imp_sorted = imp_df.sort_values('importance', ascending=False)

fig = px.bar(
    imp_sorted,
    x='importance', y='feature', orientation='h',
    color='importance', color_continuous_scale='OrRd',
    title='Feature Importances — Random Forest (Interactive)',
    labels={'importance': 'Importance Score', 'feature': 'Feature'}
)
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='white',
    coloraxis_showscale=False
)
fig.show()

---
## 12. Predict on New Objects

A self-contained `predict_staleness()` function wraps the complete pipeline. Pass a raw metadata dictionary to receive a binary prediction and probability score.

In [ ]:
def predict_staleness(raw_record: dict) -> dict:
    """
    Predict whether a single S3 object is stale.

    Parameters
    ----------
    raw_record : dict
        Expected keys: size (bytes), last_modified (str/datetime),
        last_accessed (str/datetime or None), encryption_type (str),
        is_latest (bool), delete_marker (bool), version_id (str)

    Returns
    -------
    dict : prediction (int), label (str), stale_probability (float)
    """
    # ── Parse dates ───────────────────────────────────────────────────────────
    last_mod = pd.to_datetime(raw_record.get('last_modified'))
    if last_mod.tzinfo is None:
        last_mod = last_mod.tz_localize('UTC')

    raw_accessed  = raw_record.get('last_accessed')
    never_acc_flag = 1 if raw_accessed is None else 0
    last_acc = pd.to_datetime(raw_accessed) if raw_accessed else last_mod
    if last_acc.tzinfo is None:
        last_acc = last_acc.tz_localize('UTC')

    # ── Size ──────────────────────────────────────────────────────────────────
    size_mb = max(float(raw_record.get('size', 0)), 0) / (1024 ** 2)
    size_mb = np.clip(size_mb, p01, p99)   # same Winsorization bounds as training

    # ── Encode encryption ──────────────────────────────────────────────────────
    enc_type = raw_record.get('encryption_type', 'NONE')
    enc_enc  = le.transform([enc_type])[0] if enc_type in le.classes_ else -1

    # ── Assemble feature vector ────────────────────────────────────────────────
    feats = {
        'size_log':             np.log1p(size_mb),
        'days_since_modified':  (ref_date - last_mod).days,
        'days_since_accessed':  (ref_date - last_acc).days,
        'access_to_modify_gap': (last_acc - last_mod).days,
        'is_latest':            int(raw_record.get('is_latest', True)),
        'delete_marker':        int(raw_record.get('delete_marker', False)),
        'has_encryption':       int(enc_type != 'NONE'),
        'has_version':          int(raw_record.get('version_id', 'none') != 'none'),
        'never_accessed':       never_acc_flag,
        'enc_type_encoded':     enc_enc,
        'modified_month':       last_mod.month,
        'accessed_month':       last_acc.month,
        'modified_year':        last_mod.year
    }

    X_new = pd.DataFrame([feats])[FEATURES]
    pred  = rf.predict(X_new)[0]
    prob  = rf.predict_proba(X_new)[0][1]

    return {
        'prediction':        int(pred),
        'label':             'STALE' if pred == 1 else 'ACTIVE',
        'stale_probability': round(float(prob), 4)
    }


# ── Test with two example records ──────────────────────────────────────────────
examples = [
    {   # Old object, never accessed, delete marker set → expected STALE
        'size': 500_000, 'last_modified': '2019-03-15',
        'last_accessed': None,
        'encryption_type': 'NONE', 'is_latest': False,
        'delete_marker': True, 'version_id': 'none'
    },
    {   # Recently modified and accessed, encrypted, versioned → expected ACTIVE
        'size': 12_000_000, 'last_modified': '2024-11-01',
        'last_accessed': '2025-01-15',
        'encryption_type': 'AES256', 'is_latest': True,
        'delete_marker': False, 'version_id': 'v1abc'
    }
]

for i, ex in enumerate(examples, 1):
    result = predict_staleness(ex)
    print(f'Example {i}: {result}')

---
## 13. Conclusion

### What is done
1. **Loaded and cleaned** 92,369 S3 object records with 10 columns, removing duplicates, imputing missing values by column type, and flagging never-accessed objects.
2. **Explored** class balance, size distributions, boolean flags (`is_latest`, `delete_marker`, `never_accessed`), encryption types, and bucket-level staleness patterns using Matplotlib, Seaborn, and Plotly.
3. **Detected and treated outliers** in `size_mb` using IQR detection and Winsorization.
4. **Engineered 13 features** from the raw data — most importantly `days_since_accessed`, `days_since_modified`, and `access_to_modify_gap` from the two date columns.
5. **Addressed class imbalance** with SMOTE applied to the training split only.
6. **Trained and compared** a Logistic Regression baseline and a Random Forest, using **Macro F1-Score** as the primary metric.

### Evaluation Metric Interpretation
> **Macro F1-Score** was chosen because the dataset is class-imbalanced. It averages the F1-score across both classes equally, so the model is penalised if it ignores the minority (stale) class. A high Macro F1 means the model both avoids false alarms (precision) and catches genuine stale objects (recall) for *both* class labels.

### Key Results
- **Random Forest outperformed** the Logistic Regression baseline on Macro F1 and ROC-AUC.
- **`days_since_accessed`** was the single strongest predictor — confirming that access recency is the most reliable staleness signal.
- `never_accessed`, `is_latest`, `delete_marker`, and `days_since_modified` rounded out the top features.

### Recommendations
- Integrate `predict_staleness()` into automated S3 lifecycle rules or tagging pipelines.
- Re-train monthly as access patterns evolve.
- Explore XGBoost or LightGBM for further performance gains at scale.